# EURepoC Chapter 4 Analysis Pipeline

Paste the Python pipeline code from the chat into the code cell below.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
"""
EURepoC Chapter 4 Analysis Pipeline
==================================

This script implements the updated Chapter 4 analysis framework:

4.1 Descriptive Analysis
4.2 Econometric Analysis
4.3 Semantic Analysis
4.4 Explainable Machine Learning
4.5 Graph-Based Analysis
4.6 Panel Data and Event-History Analysis
4.7 Robustness and Sensitivity Analysis

Before running:
1. Place your EURepoC Excel file in the same folder as this script, or update FILE_PATH.
2. Install required packages if missing:
   pip install pandas numpy matplotlib openpyxl statsmodels scikit-learn xgboost lightgbm catboost shap sentence-transformers bertopic umap-learn hdbscan networkx node2vec

Outputs are saved into the folder defined by OUTPUT_DIR.
"""

'\nEURepoC Chapter 4 Analysis Pipeline\n==================================\n\nThis script implements the updated Chapter 4 analysis framework:\n\n4.1 Descriptive Analysis\n4.2 Econometric Analysis\n4.3 Semantic Analysis\n4.4 Explainable Machine Learning\n4.5 Graph-Based Analysis\n4.6 Panel Data and Event-History Analysis\n4.7 Robustness and Sensitivity Analysis\n\nBefore running:\n1. Place your EURepoC Excel file in the same folder as this script, or update FILE_PATH.\n2. Install required packages if missing:\n   pip install pandas numpy matplotlib openpyxl statsmodels scikit-learn xgboost lightgbm catboost shap sentence-transformers bertopic umap-learn hdbscan networkx node2vec\n\nOutputs are saved into the folder defined by OUTPUT_DIR.\n'

In [3]:
# ============================================================
# 0. INSTALL NECESSARY LIBRARIES
# ============================================================

In [4]:
# Aggressively uninstall all relevant packages, including numpy, pandas, torch, and torchvision
!pip uninstall -y numpy numba node2vec shap lightgbm catboost umap-learn hdbscan networkx sentence-transformers transformers torch bertopic pandas matplotlib openpyxl statsmodels scikit-learn xgboost tqdm torchvision

# Pin numpy and pandas to desired compatible versions
!pip install numpy==2.0.2
!pip install pandas==2.2.2 # Pin to version compatible with google-colab

# Reinstall numba immediately after numpy
!pip install numba

# Reinstall all other necessary libraries. Let pip resolve torch version for transformers/sentence-transformers.
!pip install matplotlib openpyxl statsmodels scikit-learn xgboost lightgbm catboost shap tqdm umap-learn hdbscan networkx node2vec>=0.6.0 sentence-transformers transformers bertopic torch


Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: numba 0.65.1
Uninstalling numba-0.65.1:
  Successfully uninstalled numba-0.65.1
Found existing installation: node2vec 0.4.3
Uninstalling node2vec-0.4.3:
  Successfully uninstalled node2vec-0.4.3
Found existing installation: shap 0.52.0
Uninstalling shap-0.52.0:
  Successfully uninstalled shap-0.52.0
Found existing installation: lightgbm 4.6.0
Uninstalling lightgbm-4.6.0:
  Successfully uninstalled lightgbm-4.6.0
Found existing installation: catboost 1.2.10
Uninstalling catboost-1.2.10:
  Successfully uninstalled catboost-1.2.10
Found existing installation: umap-learn 0.5.12
Uninstalling umap-learn-0.5.12:
  Successfully uninstalled umap-learn-0.5.12
Found existing installation: hdbscan 0.8.44
Uninstalling hdbscan-0.8.44:
  Successfully uninstalled hdbscan-0.8.44
Found existing installation: networkx 3.6.1
Uninstalling networkx-3.6.1:
  Successfully unin

  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
plotnine 0.14.5 requires matplotlib>=3.8.0, which is not installed.
plotnine 0.14.5 requires statsmodels>=0.14.0, which is not installed.
momepy 0.11.0 requires networkx>=3.2, which is not installed.
momepy 0.11.0 requires tqdm>=4.65, which is not installed.
prophet 1.3.0 requires matplotlib>=2.0.0, which is not installed.
prophet 1.3.0 requires tqdm>=4.36.1, which is not installed.
sklearn-pandas 2.2.0 requires scikit-learn>=0.23.0, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
bigframes 2.42.0 requires matplotlib>=3.7.1, which is not installed.
mlxtend 0.23.4 requires matplotl

In [5]:
# ============================================================
# 0. IMPORT LIBRARIES
# ============================================================

In [6]:
import os
import warnings
warnings.filterwarnings("ignore")

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)

In [9]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [10]:
import shap
import networkx as nx

In [11]:
# Install all necessary libraries
#!pip install pandas openpyxl statsmodels scikit-learn xgboost lightgbm catboost shap sentence-transformers bertopic umap-learn hdbscan networkx node2vec

In [12]:
# ============================================================
# 1. USER SETTINGS
# ============================================================

In [13]:
FILE_PATH = "/content/drive/MyDrive/EuropoC Chapter 4 Analysis/eurepoc_global_dataset.xlsx"
OUTPUT_DIR = "chapter4_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_YEAR = 2014
END_YEAR = 2024
TEST_YEAR = 2024

FINANCE_PATTERN = r"finance|financial|bank|insurance|payment|investment|securities|credit|fintech"
STRICT_FINANCE_PATTERN = r"bank|insurance|payment|investment"

In [14]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

In [15]:
def find_col(df, keywords):
    """Find the first column containing any keyword."""
    for col in df.columns:
        col_lower = col.lower()
        for key in keywords:
            if key.lower() in col_lower:
                return col
    return None


def text_binary(x, keywords):
    """Return 1 if any keyword appears in a text value, otherwise 0."""
    x = str(x).lower()
    return int(any(k.lower() in x for k in keywords))


def temporal_split(data, year_col="year", test_year=2024):
    """Temporal train-test split."""
    train = data[data[year_col] < test_year]
    test = data[data[year_col] == test_year]
    return train, test


def safe_metric(func, *args, **kwargs):
    """Return metric if computable, otherwise np.nan."""
    try:
        return func(*args, **kwargs)
    except Exception:
        return np.nan


def evaluate_model(model, X_test, y_test, name):
    """Evaluate a binary classifier."""
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": safe_metric(roc_auc_score, y_test, prob),
        "pr_auc": safe_metric(average_precision_score, y_test, prob),
    }


def save_csv(df, name):
    path = os.path.join(OUTPUT_DIR, name)
    df.to_csv(path, index=True)
    print(f"Saved: {path}")


def save_csv_no_index(df, name):
    path = os.path.join(OUTPUT_DIR, name)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

In [16]:
# ============================================================
# 3. LOAD AND PREPARE DATA
# ============================================================

In [17]:
def load_and_prepare_data():
    df = pd.read_excel(FILE_PATH)
    print("Initial shape:", df.shape)
    print("Columns:", df.columns.tolist())

    date_col = find_col(df, ["start_date", "date", "start"])
    sector_col = find_col(df, ["receiver_category", "sector", "target sector"])
    subsector_col = find_col(df, ["receiver_subcategory", "subcategory"])
    country_col = find_col(df, ["receiver_country", "country"])
    incident_type_col = find_col(df, ["incident_type", "type"])
    actor_col = find_col(df, ["initiator_category", "actor", "initiator"])
    access_col = find_col(df, ["mitre_initial_access", "initial_access", "access"])
    description_col = find_col(df, ["description", "summary", "text"])

    print("Detected columns:")
    print("Date:", date_col)
    print("Sector:", sector_col)
    print("Subsector:", subsector_col)
    print("Country:", country_col)
    print("Incident type:", incident_type_col)
    print("Actor:", actor_col)
    print("Access:", access_col)
    print("Description:", description_col)

    required = [date_col, sector_col, country_col, incident_type_col, actor_col, access_col]
    missing = [x for x in required if x is None]
    if missing:
        raise ValueError("Some required columns were not detected. Please manually assign column names.")

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df["year"] = df[date_col].dt.year
    df["month"] = df[date_col].dt.to_period("M").astype(str)

    df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)].copy()

    df["sector_main"] = df[sector_col].astype(str).str.split(";").str[0].str.strip()
    df["country_main"] = df[country_col].astype(str).str.split(";").str[0].str.strip()
    df["incident_type_main"] = df[incident_type_col].astype(str).str.split(";").str[0].str.strip()

    if subsector_col is None:
        df["financial_target"] = df[sector_col].astype(str).str.contains(
            FINANCE_PATTERN, case=False, na=False
        ).astype(int)
    else:
        df["financial_target"] = (
            df[sector_col].astype(str).str.contains(FINANCE_PATTERN, case=False, na=False) |
            df[subsector_col].astype(str).str.contains(FINANCE_PATTERN, case=False, na=False)
        ).astype(int)

    df["data_theft"] = df[incident_type_col].apply(lambda x: text_binary(x, ["data theft", "leak", "exfiltration", "breach"]))
    df["disruption"] = df[incident_type_col].apply(lambda x: text_binary(x, ["disruption", "ddos", "outage", "sabotage"]))
    df["ransomware"] = df[incident_type_col].apply(lambda x: text_binary(x, ["ransomware"]))

    df["state_actor"] = df[actor_col].astype(str).str.contains("state", case=False, na=False).astype(int)

    df["phishing"] = df[access_col].astype(str).str.contains("phishing", case=False, na=False).astype(int)
    df["exploit"] = df[access_col].astype(str).str.contains("exploit|vulnerability", case=False, na=False).astype(int)
    df["supply_chain"] = df[access_col].astype(str).str.contains("supply", case=False, na=False).astype(int)
    df["credential"] = df[access_col].astype(str).str.contains("credential|valid account", case=False, na=False).astype(int)

    structured_vars = [
        "data_theft", "disruption", "ransomware", "state_actor",
        "phishing", "exploit", "supply_chain", "credential"
    ]

    meta = {
        "date_col": date_col,
        "sector_col": sector_col,
        "subsector_col": subsector_col,
        "country_col": country_col,
        "incident_type_col": incident_type_col,
        "actor_col": actor_col,
        "access_col": access_col,
        "description_col": description_col,
        "structured_vars": structured_vars,
    }

    save_csv_no_index(df.head(100), "00_sample_prepared_data.csv")
    return df, meta

In [18]:
# ============================================================
# 4.1 DESCRIPTIVE ANALYSIS
# ============================================================

In [19]:
# To get the filepath for your Excel file:
# 1. Upload your file to Colab (e.g., to the '/content/' directory or a folder in Google Drive).
# 2. You can then right-click on the uploaded file in the Colab file browser (left sidebar)
#    and select "Copy path" to get its full path.
# 3. Update the 'FILE_PATH' variable in cell 'LhL6FUqcPkSx' with this path.

def run_descriptive_analysis(df, meta):
    structured_vars = meta["structured_vars"]

    sector_year = df.groupby(["year", "sector_main"]).size().unstack(fill_value=0)
    sector_year.to_csv(os.path.join(OUTPUT_DIR, "4_1_1_incidents_by_year_sector.csv"))
    sector_year.plot(kind="bar", stacked=True, figsize=(14, 7))
    plt.title("Cyber Incidents by Year and Sector (2014–2024)")
    plt.xlabel("Year")
    plt.ylabel("Number of Incidents")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "4_1_1_incidents_by_year_sector.png"), dpi=300)
    plt.close()

    type_year = df.groupby(["year", "incident_type_main"]).size().unstack(fill_value=0)
    type_year.to_csv(os.path.join(OUTPUT_DIR, "4_1_2_incidents_by_year_type.csv"))
    type_year.plot(kind="bar", stacked=True, figsize=(14, 7))
    plt.title("Cyber Incidents by Year and Incident Type (2014–2024)")
    plt.xlabel("Year")
    plt.ylabel("Number of Incidents")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "4_1_2_incidents_by_year_type.png"), dpi=300)
    plt.close()

    incidents_year = df.groupby("year").size()
    incidents_year.to_csv(os.path.join(OUTPUT_DIR, "4_1_3_distribution_over_time.csv"))
    plt.figure(figsize=(10, 6))
    plt.bar(incidents_year.index, incidents_year.values)
    plt.title("Distribution of Cyber Incidents Over Time")
    plt.xlabel("Year")
    plt.ylabel("Number of Incidents")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "4_1_3_distribution_over_time.png"), dpi=300)
    plt.close()

    corr = df[["financial_target"] + structured_vars].corr()
    corr.to_csv(os.path.join(OUTPUT_DIR, "4_1_4_correlation_matrix.csv"))
    plt.figure(figsize=(10, 8))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "4_1_4_correlation_matrix.png"), dpi=300)
    plt.close()

    print("Descriptive analysis complete.")

In [20]:
# ============================================================
# 4.2 ECONOMETRIC ANALYSIS
# ============================================================

In [21]:
def run_econometric_analysis(df, meta):
    structured_vars = meta["structured_vars"]

    model_df = df[["financial_target"] + structured_vars + ["year", "country_main"]].dropna().copy()

    year_dummies = pd.get_dummies(model_df["year"], prefix="year", drop_first=True)
    X_base = pd.concat([model_df[structured_vars], year_dummies], axis=1)
    X_base = sm.add_constant(X_base).astype(float)
    y = model_df["financial_target"].astype(int)

    logit_result = sm.Logit(y, X_base).fit(cov_type="HC3")
    logit_result.summary2().tables[1].to_csv(os.path.join(OUTPUT_DIR, "4_2_1_logit_results.csv"))
    print(logit_result.summary())

    probit_result = sm.Probit(y, X_base).fit(cov_type="HC3")
    probit_result.summary2().tables[1].to_csv(os.path.join(OUTPUT_DIR, "4_2_2_probit_results.csv"))
    print(probit_result.summary())

    logit_mfx = logit_result.get_margeff()
    logit_mfx.summary_frame().to_csv(os.path.join(OUTPUT_DIR, "4_2_4_logit_marginal_effects.csv"))
    print(logit_mfx.summary())

    X_enet = model_df[structured_vars].astype(float)
    y_enet = model_df["financial_target"].astype(int)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_enet)

    elastic_net = LogisticRegressionCV(
        Cs=10,
        cv=5,
        penalty="elasticnet",
        solver="saga",
        l1_ratios=[0.1, 0.5, 0.9],
        scoring="roc_auc",
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    )
    elastic_net.fit(X_scaled, y_enet)

    enet_results = pd.DataFrame({
        "variable": X_enet.columns,
        "coefficient": elastic_net.coef_[0]
    }).sort_values("coefficient", key=abs, ascending=False)
    save_csv_no_index(enet_results, "4_2_3_elastic_net_results.csv")

    return model_df, structured_vars

In [22]:
# ============================================================
# 4.3 SEMANTIC ANALYSIS
# ============================================================

In [34]:
def run_semantic_analysis(df, meta):
    description_col = meta["description_col"]
    structured_vars = meta["structured_vars"]

    if description_col is None:
        print("No description column detected. Skipping semantic analysis.")
        return None, None, None, None

    from sentence_transformers import SentenceTransformer
    from bertopic import BERTopic

    df_text = df.copy()
    df_text["description_clean"] = (
        df_text[description_col]
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    texts = df_text["description_clean"].tolist()

    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = sbert_model.encode(texts, show_progress_bar=True)

    topic_model = BERTopic(verbose=True)
    topics, probs = topic_model.fit_transform(texts, embeddings)

    df_text["topic"] = topics

    topic_info = topic_model.get_topic_info()
    save_csv_no_index(topic_info, "4_3_1_bertopic_topic_info.csv")
    save_csv_no_index(df_text[["financial_target", "topic", "description_clean"]], "4_3_1_incident_topics.csv")

    try:
        #timestamps = df_text["year"].astype(str).tolist()
        #topics_over_time = topic_model.topics_over_time(texts, timestamps, nr_bins=10)
        date_col = meta["date_col"]

        df_text["date_for_topic"] = pd.to_datetime(
            df_text[date_col],
            errors="coerce"
        )

        valid_idx = df_text["date_for_topic"].notna()

        texts_dynamic = df_text.loc[valid_idx, "description_clean"].tolist()
        topics_dynamic = df_text.loc[valid_idx, "topic"].tolist()
        timestamps_dynamic = (
            df_text.loc[valid_idx, "date_for_topic"]
           .dt.strftime("%Y-%m-%d")
           .tolist()
        )

        topics_over_time = topic_model.topics_over_time(
            docs=texts_dynamic,
            topics=topics_dynamic,
            timestamps=timestamps_dynamic,
            nr_bins=10
        )

        save_csv_no_index(topics_over_time, "4_3_2_dynamic_bertopic.csv")
        fig = topic_model.visualize_topics_over_time(topics_over_time)
        fig.write_html(os.path.join(OUTPUT_DIR, "4_3_2_dynamic_bertopic.html"))
    except Exception as exc:
        print("Dynamic BERTopic failed:", exc)

    topic_dummies = pd.get_dummies(df_text["topic"], prefix="topic", drop_first=True)
    semantic_df = pd.concat(
        [df_text[["financial_target"] + structured_vars + ["year"]], topic_dummies],
        axis=1
    ).dropna()

    semantic_features = structured_vars + list(topic_dummies.columns)
    save_csv_no_index(semantic_df, "4_3_3_semantic_feature_dataset.csv")

    return df_text, topic_dummies, semantic_df, semantic_features

In [24]:
# ============================================================
# 4.4 EXPLAINABLE MACHINE LEARNING
# ============================================================

In [25]:
def get_ml_models():
    return {
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300, learning_rate=0.05,
            class_weight="balanced", random_state=42
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=4,
            verbose=False, random_state=42
        )
    }


def run_ml_analysis(model_df, structured_vars, semantic_df=None, semantic_features=None):
    models = get_ml_models()

    ml_a = model_df[["financial_target", "year"] + structured_vars].dropna()
    train_a, test_a = temporal_split(ml_a, test_year=TEST_YEAR)

    X_train_a = train_a[structured_vars]
    y_train_a = train_a["financial_target"]
    X_test_a = test_a[structured_vars]
    y_test_a = test_a["financial_target"]

    results_a = []
    fitted_a = {}
    for name, model in models.items():
        model.fit(X_train_a, y_train_a)
        fitted_a[name] = model
        results_a.append(evaluate_model(model, X_test_a, y_test_a, name))

    ml_results_a = pd.DataFrame(results_a)
    ml_results_a["feature_set"] = "A_structured"
    save_csv_no_index(ml_results_a, "4_4_model_A_structured_results.csv")

    ml_results_b = None
    fitted_b = {}
    if semantic_df is not None and semantic_features is not None:
        train_b, test_b = temporal_split(semantic_df, test_year=TEST_YEAR)
        X_train_b = train_b[semantic_features]
        y_train_b = train_b["financial_target"]
        X_test_b = test_b[semantic_features]
        y_test_b = test_b["financial_target"]

        results_b = []
        for name, model in models.items():
            model.fit(X_train_b, y_train_b)
            fitted_b[name] = model
            results_b.append(evaluate_model(model, X_test_b, y_test_b, name))

        ml_results_b = pd.DataFrame(results_b)
        ml_results_b["feature_set"] = "B_structured_semantic"
        save_csv_no_index(ml_results_b, "4_4_model_B_structured_semantic_results.csv")

        # SHAP for XGBoost Model B
        try:
            best_model = fitted_b["XGBoost"]
            explainer = shap.TreeExplainer(best_model)
            shap_values = explainer.shap_values(X_test_b)
            shap.summary_plot(shap_values, X_test_b, show=False)
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, "4_4_5_shap_summary.png"), dpi=300)
            plt.close()

            shap_importance = pd.DataFrame({
                "feature": X_test_b.columns,
                "mean_abs_shap": np.abs(shap_values).mean(axis=0)
            }).sort_values("mean_abs_shap", ascending=False)
            save_csv_no_index(shap_importance, "4_4_5_shap_importance.csv")
        except Exception as exc:
            print("SHAP failed:", exc)

    return ml_results_a, ml_results_b, fitted_a, fitted_b

In [26]:
# ============================================================
# 4.5 GRAPH-BASED ANALYSIS
# ============================================================

In [37]:
def run_graph_analysis(df, meta, df_text=None, topic_dummies=None):
    actor_col = meta["actor_col"]
    sector_col = meta["sector_col"]
    country_col = meta["country_col"]
    access_col = meta["access_col"]
    incident_type_col = meta["incident_type_col"]
    structured_vars = meta["structured_vars"]

    G = nx.Graph()
    graph_cols = {
        "actor": actor_col,
        "sector": sector_col,
        "country": country_col,
        "access": access_col,
        "incident_type": incident_type_col
    }

    for idx, row in df.iterrows():
        incident_node = f"incident_{idx}"
        G.add_node(incident_node, node_type="incident")
        for label, col in graph_cols.items():
            if col is not None:
                value = str(row[col]).split(";")[0].strip()
                if value and value.lower() != "nan":
                    node = f"{label}_{value}"
                    G.add_node(node, node_type=label)
                    G.add_edge(incident_node, node)

    graph_stats = pd.DataFrame({
        "metric": ["nodes", "edges"],
        "value": [G.number_of_nodes(), G.number_of_edges()]
    })
    save_csv_no_index(graph_stats, "4_5_1_graph_basic_stats.csv")

    degree_dict = dict(G.degree())
    centrality_dict = nx.degree_centrality(G)
    graph_summary = pd.DataFrame({
        "node": list(G.nodes()),
        "degree": [degree_dict[n] for n in G.nodes()],
        "degree_centrality": [centrality_dict[n] for n in G.nodes()]
    })
    save_csv_no_index(graph_summary, "4_5_2_graph_network_characteristics.csv")

    try:
        from node2vec import Node2Vec
        node2vec = Node2Vec(G, dimensions=16, walk_length=20, num_walks=50, workers=2, seed=42)
        n2v_model = node2vec.fit(window=5, min_count=1)

        embedding_dict = {node: n2v_model.wv[node] for node in G.nodes()}
        incident_embeddings = []
        for idx in df.index:
            incident_node = f"incident_{idx}"
            if incident_node in embedding_dict:
                emb = embedding_dict[incident_node]
            else:
                emb = np.zeros(16)
            incident_embeddings.append(emb)

        graph_emb_df = pd.DataFrame(
            incident_embeddings,
            index=df.index,
            columns=[f"graph_emb_{i}" for i in range(16)]
        )

        if df_text is None:
            df_text = df
        if topic_dummies is None:
            topic_dummies = pd.DataFrame(index=df.index)

        graph_model_df = pd.concat(
            [df_text[["financial_target", "year"] + structured_vars], topic_dummies, graph_emb_df],
            axis=1
        ).dropna()

        graph_features = structured_vars + list(topic_dummies.columns) + list(graph_emb_df.columns)
        save_csv_no_index(graph_model_df, "4_5_3_graph_feature_dataset.csv")

        models = get_ml_models()
        train_c, test_c = temporal_split(graph_model_df, test_year=TEST_YEAR)
        X_train_c = train_c[graph_features]
        y_train_c = train_c["financial_target"]
        X_test_c = test_c[graph_features]
        y_test_c = test_c["financial_target"]

        results_c = []
        for name, model in models.items():
            model.fit(X_train_c, y_train_c)
            results_c.append(evaluate_model(model, X_test_c, y_test_c, name))

        ml_results_c = pd.DataFrame(results_c)
        ml_results_c["feature_set"] = "C_structured_semantic_graph"
        save_csv_no_index(ml_results_c, "4_5_4_model_C_structured_semantic_graph_results.csv")
        return ml_results_c

    except Exception as exc:
        print("Node2Vec/Graph ML failed:", exc)
        return None

In [51]:
# ============================================================
# 4.6 PANEL DATA AND EVENT-HISTORY ANALYSIS
# ============================================================

In [52]:
def run_panel_event_history(df):
    panel_base = df.copy()
    panel_base["country"] = panel_base["country_main"]
    panel_base["sector"] = panel_base["sector_main"]

    incident_counts = (
        panel_base
        .groupby(["country", "sector", "month"])
        .agg(
            incident_count=("financial_target", "size"),
            state_attacks=("state_actor", "sum"),
            data_theft_count=("data_theft", "sum"),
            disruption_count=("disruption", "sum"),
            ransomware_count=("ransomware", "sum")
        )
        .reset_index()
    )

    countries = sorted(panel_base["country"].dropna().unique())
    sectors = sorted(panel_base["sector"].dropna().unique())
    months = pd.period_range(f"{START_YEAR}-01", f"{END_YEAR}-12", freq="M").astype(str)

    # Modified: Create MultiIndex and then convert to DataFrame using reset_index()
    panel_multiindex = pd.MultiIndex.from_product(
        [countries, sectors, months],
        names=["country", "sector", "month"]
    )
    panel = pd.DataFrame(index=panel_multiindex).reset_index()

    panel = panel.merge(incident_counts, on=["country", "sector", "month"], how="left")

    count_cols = ["incident_count", "state_attacks", "data_theft_count", "disruption_count", "ransomware_count"]
    panel[count_cols] = panel[count_cols].fillna(0).astype(int)

    panel["incident_occurrence"] = (panel["incident_count"] > 0).astype(int)
    panel["month_date"] = pd.to_datetime(panel["month"])
    panel["year"] = panel["month_date"].dt.year
    panel["month_num"] = panel["month_date"].dt.month
    panel["financial_sector"] = panel["sector"].str.contains(FINANCE_PATTERN, case=False, na=False).astype(int)

    #panel = panel.sort_values(["country", "sector", "month_date"])
    #group_cols = ["country", "sector"]

    panel = panel.sort_values(["country", "sector", "month_date"]).copy()

    group_cols = ["country", "sector"]

    for col in count_cols:
        panel[f"lag_{col}"] = (
            panel.groupby(group_cols)[col]
            .shift(1)
            .fillna(0)
        )

    for col in count_cols:
        panel[f"rolling3_{col}"] = (
            panel.groupby(group_cols)[col]
            .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
            .fillna(0)
        )

    save_csv_no_index(panel, "4_6_1_country_sector_month_panel.csv")

    panel_features = [
        "financial_sector",
        "lag_incident_count",
        "lag_state_attacks",
        "lag_data_theft_count",
        "lag_disruption_count",
        "lag_ransomware_count",
        "rolling3_incident_count",
        "rolling3_state_attacks",
        "rolling3_data_theft_count",
        "rolling3_disruption_count",
        "rolling3_ransomware_count"
    ]

    X_panel = panel[panel_features].copy()

    X_panel = X_panel.loc[:, X_panel.nunique() > 1]

    X_panel = X_panel.loc[:, ~X_panel.T.duplicated()]

    X_panel = sm.add_constant(X_panel, has_constant="add").astype(float)

    y_panel = panel["incident_occurrence"].astype(int)

    try:
        panel_logit = sm.Logit(y_panel, X_panel).fit(
            cov_type="HC3",
            maxiter=200
        )
    except np.linalg.LinAlgError:
        print("Singular matrix detected. Refitting without robust covariance.")
        panel_logit = sm.Logit(y_panel, X_panel).fit(
            maxiter=200
        )

    print(panel_logit.summary())

    panel_logit.summary2().tables[1].to_csv(
        os.path.join(OUTPUT_DIR, "4_6_2_event_history_logit_results.csv")
    )

    return panel

In [49]:
# ============================================================
# 4.7 ROBUSTNESS AND SENSITIVITY ANALYSIS
# ============================================================

In [50]:
def fit_logit_safely(y, X, output_name):
    # Remove zero-variance columns
    X = X.loc[:, X.nunique() > 1]

    # Remove duplicate columns
    X = X.loc[:, ~X.T.duplicated()]

    # Add constant
    X = sm.add_constant(X, has_constant="add").astype(float)
    y = y.astype(int)

    try:
        result = sm.Logit(y, X).fit(cov_type="HC3", maxiter=200)
        result.summary2().tables[1].to_csv(
            os.path.join(OUTPUT_DIR, output_name)
        )
        print(result.summary())
        print(f"Saved: {os.path.join(OUTPUT_DIR, output_name)}")
        return result

    except Exception as exc:
        print("Standard Logit failed:", exc)
        print("Using regularized Logit instead.")

        result = sm.Logit(y, X).fit_regularized(
            method="l1",
            alpha=0.01,
            maxiter=500
        )

        regularized_results = pd.DataFrame({
            "variable": X.columns,
            "coefficient": result.params
        })

        regularized_results.to_csv(
            os.path.join(OUTPUT_DIR, output_name),
            index=False
        )

        print(f"Saved regularized results: {os.path.join(OUTPUT_DIR, output_name)}")
        return result

def run_robustness(df, meta, model_df, structured_vars):
    sector_col = meta["sector_col"]
    subsector_col = meta["subsector_col"]

    # 4.7.1 Alternative strict financial-sector definition
    if subsector_col is None:
        df["financial_target_strict"] = df[sector_col].astype(str).str.contains(
            STRICT_FINANCE_PATTERN, case=False, na=False
        ).astype(int)
    else:
        df["financial_target_strict"] = (
            df[sector_col].astype(str).str.contains(STRICT_FINANCE_PATTERN, case=False, na=False) |
            df[subsector_col].astype(str).str.contains(STRICT_FINANCE_PATTERN, case=False, na=False)
        ).astype(int)

    robust_df = df[["financial_target_strict"] + structured_vars + ["year"]].dropna()

    X_robust = pd.concat(
        [
            robust_df[structured_vars],
            pd.get_dummies(robust_df["year"], prefix="year", drop_first=True)
        ],
        axis=1
    )

    y_robust = robust_df["financial_target_strict"].astype(int)

    fit_logit_safely(
        y_robust,
        X_robust,
        "4_7_1_strict_finance_robustness.csv"
    )

    # 4.7.2 Europe-only subsample analysis
    europe_countries = [
        "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic",
        "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary",
        "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
        "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia",
        "Spain", "Sweden", "Norway", "Switzerland", "United Kingdom"
    ]

    eu_df = df[df["country_main"].isin(europe_countries)].copy()
    eu_model = eu_df[["financial_target"] + structured_vars + ["year"]].dropna()

    if len(eu_model) > 0 and eu_model["financial_target"].nunique() == 2:
        X_eu = pd.concat(
            [
                eu_model[structured_vars],
                pd.get_dummies(eu_model["year"], prefix="year", drop_first=True)
            ],
            axis=1
        )

        y_eu = eu_model["financial_target"].astype(int)

        fit_logit_safely(
            y_eu,
            X_eu,
            "4_7_2_europe_only_logit.csv"
        )

    else:
        print("Europe-only model skipped because the subsample is empty or has one class only.")

    # 4.7.3 Expanding-window validation using XGBoost structured variables
    def expanding_window_evaluation(data, features, target, years):
        output = []

        for test_year in years:
            train = data[data["year"] < test_year]
            test = data[data["year"] == test_year]

            if len(train) == 0 or len(test) == 0:
                continue

            if train[target].nunique() < 2 or test[target].nunique() < 2:
                continue

            X_train = train[features]
            y_train = train[target]
            X_test = test[features]
            y_test = test[target]

            model = XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42
            )

            model.fit(X_train, y_train)

            metrics = evaluate_model(
                model,
                X_test,
                y_test,
                f"XGBoost_{test_year}"
            )

            metrics["test_year"] = test_year
            output.append(metrics)

        return pd.DataFrame(output)

    ml_a = model_df[["financial_target", "year"] + structured_vars].dropna()

    expanding_results = expanding_window_evaluation(
        ml_a,
        structured_vars,
        "financial_target",
        years=[2019, 2020, 2021, 2022, 2023, 2024]
    )

    save_csv_no_index(
        expanding_results,
        "4_7_3_expanding_window_validation.csv"
    )

    return expanding_results

In [38]:
# ============================================================
# MAIN EXECUTION
# ============================================================

In [53]:
if __name__ == "__main__":
    df, meta = load_and_prepare_data()

    run_descriptive_analysis(df, meta)

    model_df, structured_vars = run_econometric_analysis(df, meta)

    df_text, topic_dummies, semantic_df, semantic_features = run_semantic_analysis(df, meta)

    ml_results_a, ml_results_b, fitted_a, fitted_b = run_ml_analysis(
        model_df, structured_vars, semantic_df, semantic_features
    )

    ml_results_c = run_graph_analysis(df, meta, df_text, topic_dummies)

    # Combine model comparisons if available
    comparisons = []
    if ml_results_a is not None:
        comparisons.append(ml_results_a)
    if ml_results_b is not None:
        comparisons.append(ml_results_b)
    if ml_results_c is not None:
        comparisons.append(ml_results_c)
    if comparisons:
        model_comparison = pd.concat(comparisons, axis=0)
        save_csv_no_index(model_comparison, "4_5_5_incremental_model_comparison.csv")

    panel = run_panel_event_history(df)

    run_robustness(df, meta, model_df, structured_vars)

    print("\nAll analyses complete. Check the output folder:", OUTPUT_DIR)

Initial shape: (3414, 85)
Columns: ['incident_id', 'name', 'description', 'start_date', 'end_date', 'inclusion_criterion', 'inclusion_criterion_subcode', 'source_disclosure', 'incident_type', 'receiver_name', 'receiver_country', 'receiver_country_alpha_2_code', 'receiver_regions', 'receiver_category', 'receiver_subcategory', 'initiator_name', 'initiator_country', 'initiator_alpha_2', 'initiator_category', 'initiator_subcategory', 'number_attributions', 'attribution_id', 'attribution_date', 'attribution_type', 'attribution_basis', 'attributing_actor', 'attributing_company', 'attributing_country', 'settled_initiator', 'initiator_country.1', 'initiator_category.1', 'attribution_source_url', 'cyber_conflict_issue', 'offline_conflict_issue', 'offline_conflict_name', 'offline_conflict_intensity', 'offline_conflict_intensity_subcode', 'number_political_responses', 'political_response_date', 'political_response_type', 'political_response_subtype', 'political_response_responding_country', 'poli

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

2026-06-19 00:16:25,775 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-19 00:16:45,565 - BERTopic - Dimensionality - Completed ✓
2026-06-19 00:16:45,566 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-19 00:16:45,681 - BERTopic - Cluster - Completed ✓
2026-06-19 00:16:45,687 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-19 00:16:46,001 - BERTopic - Representation - Completed ✓


Saved: chapter4_outputs/4_3_1_bertopic_topic_info.csv
Saved: chapter4_outputs/4_3_1_incident_topics.csv


10it [00:01,  8.88it/s]


Saved: chapter4_outputs/4_3_2_dynamic_bertopic.csv
Saved: chapter4_outputs/4_3_3_semantic_feature_dataset.csv
[LightGBM] [Info] Number of positive: 162, number of negative: 1896
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000618 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14
[LightGBM] [Info] Number of data points in the train set: 2058, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

Computing transition probabilities:   0%|          | 0/3526 [00:00<?, ?it/s]

Saved: chapter4_outputs/4_5_3_graph_feature_dataset.csv
[LightGBM] [Info] Number of positive: 162, number of negative: 1896
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001929 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4146
[LightGBM] [Info] Number of data points in the train set: 2058, number of used features: 49
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Saved: chapter4_outputs/4_5_4_model_C_structured_semantic_graph_results.csv
Saved: chapter4_outputs/4_5_5_incremental_model_comparison.csv
Saved: chapter4_outputs/4_6_1_country_sector_month_panel.csv
Optimization terminated successfully.
         Current function value: 0.037034
         Iterations 10
                            Logit Regression Results                           
Dep. Variable:     incident_occurrence   No. Observations:               

In [47]:
# ============================================================
# DOWNLOAD OUTPUT FILES
# ============================================================

In [48]:
import shutil
from google.colab import files

# Create a zip archive of the output directory
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)

# Download the zip file
files.download(f'{OUTPUT_DIR}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>